# PMAPS Workshop: IDAES-GTEP, Session 2

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/agmoore4/idaes-gtep.git/pmaps?urlpath=%2Fdoc%2Ftree%2Fdocs%2Fsource%2Ftutorials%2F123bus%2Ftutorial_123bus.ipynb)

Welcome! In this tutorial, we will demonstrate using IDAES Generation and Transmission Expansion Planning (GTEP) with a more complex case--a 123-bus system in Texas.

#### Read in data

In [1]:
# suppressing some logs/warnings
import logging
import warnings

logging.getLogger().setLevel(logging.ERROR)
warnings.filterwarnings("ignore")

In [2]:
from pathlib import Path
from IPython.display import Markdown, display
from urllib.parse import quote

def display_markdown_of_paths(paths):
    display(Markdown("\n".join(
        f"- [{f.name}]({quote(f.as_posix())})"
        for f in paths
    )))

data_path = Path("../../../../gtep/data/123_Bus_Resil_Week")
display_markdown_of_paths(data_path.glob("*.csv"))

- [branch.csv](../../../../gtep/data/123_Bus_Resil_Week/branch.csv)
- [bus.csv](../../../../gtep/data/123_Bus_Resil_Week/bus.csv)
- [Bus_data_gen_weights_mappings.csv](../../../../gtep/data/123_Bus_Resil_Week/Bus_data_gen_weights_mappings.csv)
- [bus_to_county.csv](../../../../gtep/data/123_Bus_Resil_Week/bus_to_county.csv)
- [candidate_generators_initial_list.csv](../../../../gtep/data/123_Bus_Resil_Week/candidate_generators_initial_list.csv)
- [candidate_generators_longer_list.csv](../../../../gtep/data/123_Bus_Resil_Week/candidate_generators_longer_list.csv)
- [county_fips_match.csv](../../../../gtep/data/123_Bus_Resil_Week/county_fips_match.csv)
- [DAY_AHEAD_load.csv](../../../../gtep/data/123_Bus_Resil_Week/DAY_AHEAD_load.csv)
- [DAY_AHEAD_renewables.csv](../../../../gtep/data/123_Bus_Resil_Week/DAY_AHEAD_renewables.csv)
- [gen small_candidates.csv](../../../../gtep/data/123_Bus_Resil_Week/gen%20small_candidates.csv)
- [gen.csv](../../../../gtep/data/123_Bus_Resil_Week/gen.csv)
- [gen_week1runs.csv](../../../../gtep/data/123_Bus_Resil_Week/gen_week1runs.csv)
- [may_20.csv](../../../../gtep/data/123_Bus_Resil_Week/may_20.csv)
- [may_24.csv](../../../../gtep/data/123_Bus_Resil_Week/may_24.csv)
- [REAL_TIME_load.csv](../../../../gtep/data/123_Bus_Resil_Week/REAL_TIME_load.csv)
- [REAL_TIME_renewables.csv](../../../../gtep/data/123_Bus_Resil_Week/REAL_TIME_renewables.csv)
- [reserves.csv](../../../../gtep/data/123_Bus_Resil_Week/reserves.csv)
- [simulation_objects.csv](../../../../gtep/data/123_Bus_Resil_Week/simulation_objects.csv)
- [timeseries_pointers.csv](../../../../gtep/data/123_Bus_Resil_Week/timeseries_pointers.csv)

In [3]:
from gtep.gtep_data import ExpansionPlanningData

data_object = ExpansionPlanningData(
    stages=1,
    num_reps=1,
    num_commit=1,
    num_dispatch=1,
)
data_object.load_prescient(data_path)

Interactive Python mode detected; using default matplotlib backend for plotting.
Setting default t0 state in RTS-GMLC parser


#### Cost data

In [4]:
from gtep.gtep_data_processing import DataProcessing

bus_data_path = Path(
    "../../../../gtep/data/costs/Bus_data_gen_weights_mappings.csv"
)
cost_data_path = Path(
    "../../../../gtep/data/costs/2022_v3_Annual_Technology_Baseline_Workbook_Mid-year_update_2-15-2023_Clean.xlsx"
)
ng_cost_path = Path(
    "../../../../gtep/data/costs/Total_Energy_Supply_Disposition_and_Price_Summary.csv"
)

candidate_gens = [
    "Natural Gas_FE",
    "Solar - Utility PV",
    "Land-Based Wind",
]

cost_data = DataProcessing()
cost_data.load_gen_data(
    bus_data_path=bus_data_path,
    cost_data_path=cost_data_path,
    ng_cost_path=ng_cost_path,
    candidate_gens=candidate_gens,
)

In [5]:
from gtep.gtep_model import ExpansionPlanningModel
from pyomo.environ import SolverFactory, TransformationFactory

mod_object = ExpansionPlanningModel(
    data=data_object,
    cost_data=cost_data,
    config={"scale_loads": False},
)
mod_object.create_model()

TransformationFactory("gdp.bigm").apply_to(mod_object.model)
opt = SolverFactory("highs")

result = opt.solve(mod_object.model, tee=True)

[    0.00] Creating GTEP Model
Running HiGHS 1.13.1 (git hash: 1d267d9): Copyright (c) 2026 under MIT licence terms
MIP has 8044 rows; 7092 cols; 21884 nonzeros; 3450 integer variables (3088 binary)
Coefficient ranges:
  Matrix  [1e+00, 3e+06]
  Cost    [1e+00, 4e+09]
  Bound   [1e+00, 4e+03]
  RHS     [3e-01, 3e+06]
Presolving model
4222 rows, 3402 cols, 11594 nonzeros  0s
3516 rows, 2643 cols, 9599 nonzeros  0s
3315 rows, 2413 cols, 9147 nonzeros  0s
Presolve reductions: rows 3315(-4729); columns 2413(-4679); nonzeros 9147(-12737) 

Solving MIP model with:
   3315 rows
   2413 cols (1480 binary, 0 integer, 0 implied int., 933 continuous, 0 domain fixed)
   9147 nonzeros

Src: B => Branching; C => Central rounding; F => Feasibility pump; H => Heuristic;
     I => Shifting; J => Feasibility jump; L => Sub-MIP; P => Empty MIP; R => Randomized rounding;
     S => Solve LP; T => Evaluate node; U => Unbounded; X => User solution; Y => HiGHS solution;
     Z => ZI Round; l => Trivial lower;

In [6]:
from pyomo.environ import value

for i in mod_object.model.stages:
    print("-" * 50)
    print(f"INVESTMENT STAGE {i}")

    print("Which thermal generators are operational or extended:")
    for thermal_generator in mod_object.model.thermalGenerators:
        print(
            thermal_generator,
            " " * (30 - len(thermal_generator)),
            (
                value(mod_object.model.investmentStage[i].genOperational[thermal_generator].indicator_var)
                or value(mod_object.model.investmentStage[i].genExtended[thermal_generator].indicator_var)
            ),
        )

    print("Renewables operational or extended generation capacity:")
    for renewable_generator in mod_object.model.renewableGenerators:
        print(
            renewable_generator,
            " " * (30 - len(renewable_generator)),
            (
                value(mod_object.model.investmentStage[i].renewableOperational[renewable_generator])
                + value(mod_object.model.investmentStage[i].renewableExtended[renewable_generator])
            ),
        )

--------------------------------------------------
INVESTMENT STAGE 1
Which thermal generators are operational or extended:
1                               True
2                               True
3                               True
4                               True
5                               True
6                               True
7                               True
8                               True
10                              True
11                              True
12                              True
13                              True
14                              True
15                              True
16                              True
18                              True
25                              True
26                              True
27                              True
28                              True
44                              True
45                              True
47                              True
48                       

In [9]:
from copy import copy
from IPython.display import display, HTML

def display_plotly_as_html(fig, title=None):
    if title is not None:
        fig = copy(fig)
        fig.update_layout(title=title)
    fig_html = fig.to_html(full_html=False, include_plotlyjs="cdn")
    return display(HTML(fig_html))

In [11]:
from gtep.gtep_solution import ExpansionPlanningSolution

# create solution object and write out to json
soln = ExpansionPlanningSolution(data_path)
soln_path = Path("soln")
soln.save_results_in_json_files(mod_object, soln_path)

# perform plotting
rep_days = [
    value(mod_object.model.representativeDate[idx])
    for idx in mod_object.model.representativeDate
]
pie = soln.create_plots("combined", soln_path, data_path, "piechart", savefig=False)
stackgraph = soln.create_stackgraph(soln_path, rep_days, savefig=False)

The following files have been created in the directory 'soln':
 - soln/renewable_investments.json
 - soln/dispatchable_investments.json
 - soln/load_shed.json
 - soln/costs.json
 - soln/flows.json
 - soln/generation.json
 - soln/curtailment.json
 - soln/loads.json
 - soln/reserves.json
 - soln/charging.json
 - soln/discharging.json


In [12]:
display_plotly_as_html(pie)
display_plotly_as_html(stackgraph)